<a href="https://colab.research.google.com/github/SivaSwetha-baba/my_first_repo/blob/main/module1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time
import sqlite3
import pandas as pd
import numpy as np
import os



# ---------------------------------------------------
# 1. Website URL
# ---------------------------------------------------

BASE_URL = "https://books.toscrape.com/"

headers = {
    "User-Agent": "Mozilla/5.0"
}


# ---------------------------------------------------
# 2. Get the homepage
# ---------------------------------------------------

response = requests.get(BASE_URL, headers=headers, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


# ---------------------------------------------------
# 3. Find book categories
# ---------------------------------------------------

category_section = soup.select_one(".side_categories")

category_links = category_section.select("ul li ul li a")

categories = []

for link in category_links:
    category_name = link.text.strip()
    category_url = urljoin(BASE_URL, link["href"])

    categories.append({
        "name": category_name,
        "url": category_url
    })

# ---------------------------------------------------
# 4. Select 3 categories
# ---------------------------------------------------

selected_categories = categories[:3]

print("\nCategories we will scrape:")

for category in selected_categories:
    print("- ", category["name"])


# ---------------------------------------------------
# 5. Scrape books
# ---------------------------------------------------

all_books = []


for category in selected_categories:

    category_name = category["name"]
    category_url = category["url"]

    print("\nScraping category:", category_name)

    page_number = 1

    while True:

        # First page has index.html.
        # Following pages have page-2.html, page-3.html, etc.l
        if page_number == 1:
            page_url = category_url
        else:
            page_url = category_url.replace(
                "index.html",
                f"page-{page_number}.html"
            )

        print("  Scraping:", page_url)

        response = requests.get(
            page_url,
            headers=headers,
            timeout=10
        )

        # Stop if the page doesn't exist
        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        # Find all books on this page
        books = soup.select("article.product_pod")

        if not books:
            break

        # ------------------------------------------------
        # 6. Extract information from every book
        # ------------------------------------------------

        for book in books:

            # Title
            title = book.select_one("h3 a")["title"]

            # Price
            price = book.select_one("p.price_color").text.replace("Â£", "").strip()
            price=float(price)
            # Star rating
            rating_element = book.select_one("p.star-rating")



            #print(f"Ratingggggggggggggggggggggggggggggg {rating_element}")

            #df["rating"] = df["star_rating"].apply(parse_rating)



            rating_classes = rating_element.get("class")

            # Example:
            # ['star-rating', 'Three']
            star_rating = rating_classes[1]

            rating_map = {
                "One": 1,
                "Two": 2,
                "Three": 3,
                "Four": 4,
                "Five": 5
                        }
            rating = rating_map.get(star_rating)
            availability_text= book.select_one(
                "p.availability"
            ).text.strip()

            text = str(availability_text).lower()
            availability_status=False
            if "in stock" in text:
                 availability_status=True
            else:
              if "out of stock" in text:
                availability_status=False
           #df["in_stock"] = df["availability"].apply(parse_availability)
           # Save the book
           # df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())

            #df["rating"] = df["rating"].fillna(df["rating"].median()).round().astype(int)
            all_books.append({
                "title": title,
                "price_gbp": price,
                "star_rating": rating,
                "availability": availability_status,
                "category": category_name
            })

        print("  Books found:", len(books))

# ---------------------------------------------------------
# 4. Median imputation for numeric fields
# ---------------------------------------------------------

# price_gbp and rating are numeric fields.
# Replace invalid/missing values with their respective medians.




# ---------------------------------------------------------
# 5. Handle invalid availability rows
# ---------------------------------------------------------

# Availability is categorical/boolean rather than numeric.
# If it cannot be parsed, drop that row because we cannot
# reliably determine whether the book is in stock.

#df = df.dropna(subset=["in_stock"])

#df["in_stock"] = df["in_stock"].astype(bool)

        # ------------------------------------------------
        # 7. Check whether there is another page
        # ------------------------------------------------

        next_button = soup.select_one("li.next a")

        if next_button is None:
            break

        page_number += 1

        # Small delay between requests
        time.sleep(1)


# ---------------------------------------------------
# 8. Convert data into a DataFrame
# ---------------------------------------------------

df = pd.DataFrame(all_books)


# ---------------------------------------------------
# 9. Display results
# ---------------------------------------------------

print("\n--------------------------------")
print("SCRAPING COMPLETE")
print("--------------------------------")

print("Total books scraped:", len(df))

print("\nFirst 10 books:")
print(df.head(10))


# ---------------------------------------------------
# 10. Save to CSV
# ---------------------------------------------------

df.to_csv(
    "books_scraped.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved as: books_scraped.csv")




# ============================================================
# 10. Convert GBP to INR
# ============================================================

GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

print(df[["title", "price_gbp", "price_inr"]].head())


# ============================================================
# 11. Create SQLite database
# ============================================================

# Close any existing connection to prevent "database is locked" errors from previous runs
if 'conn' in locals() and isinstance(conn, sqlite3.Connection):
    conn.close()

# Ensure the database file is not locked from previous runs by deleting it if it exists
if os.path.exists("books.db"):
    os.remove("books.db")

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

# Set isolation_level to None for autocommit mode.
# This is crucial for DDL statements (like DROP TABLE) to commit immediately
# and helps prevent "database is locked" errors when modifying schema.
conn.isolation_level = None

# Drop tables if they exist to ensure a clean state for each run
# These drop statements are technically redundant if we delete the file, but are harmless.
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")
# No need for conn.commit() here for DDL as isolation_level = None implies autocommit.

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Tables creation checkingggggggggggggggggggg")
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())
# No need for conn.commit() and cursor.close()/re-create here,
# as tables are dropped and recreated at the beginning

#Inserting category valus from bookscraped.csv

categories = df["category"].dropna().unique()
print(f"Printing categoriessssssssssss {categories}")
for category in categories:
    cursor.execute(
        """
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )
conn.commit() # Commit after category insertions

print(pd.read_sql("SELECT * FROM categories", conn))

category_df = pd.read_sql(
    "SELECT category_id, category_name FROM categories",
    conn
)

print(category_df)

df = df.merge(
    category_df,
    left_on="category",
    right_on="category_name",
    how="left"
)

print(df[["title", "category", "category_id"]].head())


#inserting values for books table from bookscraped.csv


for _, row in df.iterrows():

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["star_rating"]),
        int(row["availability"]),
        int(row["category_id"])
    ))

# Moved commit outside the loop for efficiency and to prevent locking issues
conn.commit()

# No longer reading books_check inside the loop
books_check = pd.read_sql(
    "SELECT * FROM books",
    conn
)

print(books_check.head())
print("Number of books:\n", len(books_check))


print("Categories:\n")
print(pd.read_sql("SELECT * FROM categories", conn))

print("\nBooks:\n")
print(pd.read_sql("SELECT * FROM books LIMIT 10", conn))

# Final commit and close at the end of the cell
conn.commit()
conn.close()

query1 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp > 30
"""

# Re-establish connection for queries as it was closed after inserts
conn = sqlite3.connect("books.db")
result1 = pd.read_sql(query1, conn)

print("QUERY 1:")
print(query1)
print(result1)

query2 = """
SELECT title, price_gbp, price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

print("QUERY 2:")
print(query2)
print(result2)

query3 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name
"""

result3 = pd.read_sql(query3, conn)

print("QUERY 3:")
print(query3)
print(result3)

query4 = """
SELECT title, price_gbp, rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp
"""

result4 = pd.read_sql(query4, conn)

print("QUERY 4:")
print(query4)
print(result4)


query5 = """
SELECT title, price_gbp, rating
FROM books
WHERE category_id IN (
    SELECT category_id
    FROM categories
    WHERE category_name IN ('Travel', 'Mystery & Thriller')
)
ORDER BY rating DESC
"""

result5 = pd.read_sql(query5, conn)

print("QUERY 5:")
print(query5)
print(result5)

#Join Queryyy

join_query = """
SELECT
    c.category_name,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY c.category_name, b.rating DESC, b.title
"""

join_result = pd.read_sql(join_query, conn)

print("JOIN QUERY:")
print(join_query)
print(join_result)

queries = {
    "query1_select_where": query1,
    "query2_order_limit": query2,
    "query3_distinct": query3,
    "query4_between": query4,
    "query5_in": query5,
    "query6_join": join_query
}

query_outputs = {}

for name, query in queries.items():
    query_outputs[name] = pd.read_sql(query, conn)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("SQL:")
    print(query)
    print("\nOUTPUT:")
    print(query_outputs[name])

 #Reading two query results into pandas

df_query1 = pd.read_sql(query1, conn)
df_query2 = pd.read_sql(query2, conn)

print("Query 1 DataFrame:")
print(df_query1)

print("\nQuery 2 DataFrame:")
print(df_query2)


#Reproduce the Json using pd.merge


books_df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "star_rating",  # Corrected column name
        "availability", # Corrected column name
        "category_id"
    ]
].copy()

categories_df = category_df[ # Corrected variable name from category_map to category_df
    [
        "category_id",
        "category_name"
    ]
].copy()


merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

merge_result = merge_result[
    [
        "category_name",
        "title",
        "price_gbp",
        "price_inr",
        "star_rating", # Corrected column name
        "availability" # Corrected column name
    ]
]

merge_result = merge_result.sort_values(
    ["category_name", "star_rating", "title"], # Corrected column name
    ascending=[True, False, True]
).reset_index(drop=True)

print("PANDAS MERGE RESULT:")
print(merge_result)


sql_join_compare = join_result[
    [
        "category_name",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock"
    ]
].copy()

sql_join_compare = sql_join_compare.sort_values(
    ["category_name", "rating", "title"],
    ascending=[True, False, True]
).reset_index(drop=True)

# To compare, rename columns in merge_result to match sql_join_compare
merge_result_for_comparison = merge_result.rename(columns={'star_rating': 'rating', 'availability': 'in_stock'})

'''print(
    "Are SQL JOIN and pandas merge equivalent?",
    sql_join_compare.equals(merge_result_for_comparison)
)'''
print(
    np.allclose(
        sql_join_compare["price_inr"],
        merge_result_for_comparison["price_inr"]
    ))

# Close connection after all queries are done
conn.close()


Categories we will scrape:
-  Travel
-  Mystery
-  Historical Fiction

Scraping category: Travel
  Scraping: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
  Books found: 11

Scraping category: Mystery
  Scraping: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
  Books found: 20
  Scraping: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
  Books found: 12

Scraping category: Historical Fiction
  Scraping: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
  Books found: 20
  Scraping: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
  Books found: 6

--------------------------------
SCRAPING COMPLETE
--------------------------------
Total books scraped: 69

First 10 books:
                                               title  price_gbp  star_rating  \
0                            It's Only the Himalayas      45.17            2   
1  Full Moon